In [4]:
import urllib.request
import urllib.error
import os
from datetime import datetime
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_single_file(year, day, base_url, download_path, max_retries=3):
    """Скачивание файла с проверкой версий R01, R02, R03"""
    day_str = f"{year}{day:03d}"
    base_filename = f"TIDI_PB_{day_str}_P0100_S0450_D011"
    
    # Список версий для проверки в порядке приоритета
    versions = ['R01', 'R02', 'R03']
    
    for version in versions:
        filename = f"{base_filename}_{version}.VEC.gz"
        url = base_url + filename
        filepath = os.path.join(download_path, str(year), filename)
        
        # Проверяем, не скачан ли уже файл
        if os.path.exists(filepath):
            return f"{filename} - уже существует"
        
        # Пробуем скачать
        for attempt in range(max_retries):
            try:
                urllib.request.urlretrieve(url, filepath)
                return f"{filename} - OK"
            except urllib.error.HTTPError as e:
                if e.code == 404:
                    # Файл не найден, пробуем следующую версию
                    break
                else:
                    if attempt == max_retries - 1:
                        return f"{filename} - ОШИБКА: {e}"
                    time.sleep(1)
            except Exception as e:
                if attempt == max_retries - 1:
                    return f"{filename} - ОШИБКА: {e}"
                time.sleep(1)
    
    # Если ни одна версия не найдена
    return f"День {day_str} - НЕТ ДОСТУПНЫХ ФАЙЛОВ (R01, R02, R03)"

def download_tidi_fast(year, download_path, max_workers=5):
    """Быстрое скачивание с использованием многопоточности"""
    is_leap_year = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)
    days_in_year = 366 if is_leap_year else 365
    
    full_path = os.path.join(download_path, str(year))
    os.makedirs(full_path, exist_ok=True)
    
    base_url = f"https://cdaweb.gsfc.nasa.gov/pub/data/timed/tidi/vector/{year}/"
    
    print(f"Скачивание {days_in_year} файлов для {year} года используя {max_workers} потоков...")
    print(f"Будут проверяться версии: R01, R02, R03")
    print("-" * 60)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_single_file, year, day, base_url, download_path): day 
                  for day in range(1, days_in_year + 1)}
        
        for i, future in enumerate(as_completed(futures), 1):
            result = future.result()
            print(f"[{i:3d}/{days_in_year}] {result}")

# Использование - просто меняйте год и max_workers при необходимости
download_tidi_fast(2018, r"C:/Users/Maks/Desktop/Jupyter/satellite_data/TIDI", max_workers=10)

Скачивание 365 файлов для 2017 года используя 10 потоков...
Будут проверяться версии: R01, R02, R03
------------------------------------------------------------
ERROR! Session/line number was not unique in database. History logging moved to new session 638
[  1/365] TIDI_PB_2017001_P0100_S0450_D011_R01.VEC.gz - OK
[  2/365] TIDI_PB_2017002_P0100_S0450_D011_R01.VEC.gz - OK
[  3/365] TIDI_PB_2017003_P0100_S0450_D011_R01.VEC.gz - OK
[  4/365] TIDI_PB_2017011_P0100_S0450_D011_R01.VEC.gz - OK
[  5/365] TIDI_PB_2017004_P0100_S0450_D011_R01.VEC.gz - OK
[  6/365] TIDI_PB_2017005_P0100_S0450_D011_R01.VEC.gz - OK
[  7/365] TIDI_PB_2017012_P0100_S0450_D011_R01.VEC.gz - OK
[  8/365] TIDI_PB_2017013_P0100_S0450_D011_R01.VEC.gz - OK
[  9/365] TIDI_PB_2017006_P0100_S0450_D011_R01.VEC.gz - OK
[ 10/365] TIDI_PB_2017007_P0100_S0450_D011_R01.VEC.gz - OK
[ 11/365] TIDI_PB_2017014_P0100_S0450_D011_R01.VEC.gz - OK
[ 12/365] TIDI_PB_2017015_P0100_S0450_D011_R01.VEC.gz - OK
[ 13/365] TIDI_PB_2017017_P0100_S04